# ROI extraction and stimulus alignment

This notebook builds bilateral EBA, FFA, and PPA response matrices for each subject and
records the stimulus order used downstream. Paths come from `algonauts_rsa.config`; Colab
Drive remains supported as a fallback, while local runs use environment variables.


In [ ]:
# Optional Colab setup
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass


In [ ]:
from algonauts_rsa.config import get_paths

PATHS = get_paths()
DATA_ROOT = PATHS.data_root
print("DATA_ROOT exists?", DATA_ROOT.exists(), DATA_ROOT)

left = DATA_ROOT / "subj01/training_split/training_fmri/lh_training_fmri.npy"
right = DATA_ROOT / "subj01/training_split/training_fmri/rh_training_fmri.npy"
print("LH file exists?", left.exists())
print("RH file exists?", right.exists())


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from algonauts_rsa.config import get_paths
from algonauts_rsa.data import (
    get_roi_label_map,
    load_training_fmri,
    parse_training_image_meta,
    roi_bool_from_labels,
    validate_alignment,
)

PATHS = get_paths()
DATA_ROOT = PATHS.data_root
OUT_BASE = PATHS.multi_roi_dir

DEBUG = False
DEBUG_SUBJECT = "subj01"
DEBUG_ROI_KEY = "FFA"
DEBUG_N_IMAGES = None
SUBJECTS = [f"subj{idx:02d}" for idx in range(1, 9)]
ROI_SPECS = {
    "EBA": ("floc-bodies", ["EBA"]),
    "FFA": ("floc-faces", ["FFA-1", "FFA-2"]),
    "PPA": ("floc-places", ["PPA"]),
}
(OUT_BASE / "meta").mkdir(parents=True, exist_ok=True)

# ---------- RUN ----------
rows = []
for subj in SUBJECTS:
    sub_dir = DATA_ROOT / subj
    if not sub_dir.exists():
        print(f"[SKIP] {subj}: folder not found -> {sub_dir}")
        continue

    try:
        X_lh, X_rh = load_training_fmri(sub_dir)
    except Exception as e:
        print(f"[SKIP] {subj}: {e}")
        continue

    n_imgs_all = X_lh.shape[0]
    # Limit images in DEBUG mode (keeps alignment with sorted filenames)
    if DEBUG and DEBUG_N_IMAGES is not None:
        n_keep = min(n_imgs_all, int(DEBUG_N_IMAGES))
        X_lh = X_lh[:n_keep]
        X_rh = X_rh[:n_keep]
    else:
        n_keep = n_imgs_all

    # Save image meta once per subject (respecting any DEBUG slice)
    try:
        meta_df = parse_training_image_meta(sub_dir / "training_split" / "training_images", n_keep)
        validate_alignment(meta_df, X_lh)
        meta_df.to_csv(OUT_BASE / "meta" / f"{subj}_train_images_meta.csv", index=False)
    except Exception as e:
        print(f"[WARN] Could not save image meta for {subj}: {e}")

    for roi_key, (roi_class, labels) in ROI_SPECS.items():
        out_dir = OUT_BASE / roi_key
        out_dir.mkdir(parents=True, exist_ok=True)

        try:
            lh_mask, lh_map = get_roi_label_map(sub_dir, roi_class, "lh")
            rh_mask, rh_map = get_roi_label_map(sub_dir, roi_class, "rh")

            lh_idx = roi_bool_from_labels(lh_mask, lh_map, labels)
            rh_idx = roi_bool_from_labels(rh_mask, rh_map, labels)

            X_roi_lh = X_lh[:, lh_idx] if lh_idx.any() else np.empty((n_keep, 0))
            X_roi_rh = X_rh[:, rh_idx] if rh_idx.any() else np.empty((n_keep, 0))
            X_roi = np.hstack([X_roi_lh, X_roi_rh])

            rows.append({
                "subject": subj,
                "roi_key": roi_key,
                "roi_class": roi_class,
                "labels": ",".join(labels),
                "lh_vertices": int(lh_idx.sum()),
                "rh_vertices": int(rh_idx.sum()),
                "bilat_vertices": int(lh_idx.sum() + rh_idx.sum()),
                "n_images": int(n_keep),
                "saved": bool(X_roi.shape[1] > 0)
            })

            if X_roi.shape[1] == 0:
                print(f"[WARN] {subj} {roi_key}: no vertices; skipping save.")
                continue

            np.save(out_dir / f"{subj}.npy", X_roi)
            print(f"[OK] {subj} {roi_key}: {X_roi.shape} -> {out_dir / f'{subj}.npy'}")

        except FileNotFoundError as e:
            print(f"[SKIP] {subj} {roi_key}: {e}")
        except Exception as e:
            print(f"[ERR ] {subj} {roi_key}: {e}")

# Summary CSV
summary = pd.DataFrame(rows)
summary_path = OUT_BASE / "summary_vertex_counts.csv"
summary.to_csv(summary_path, index=False)
print(f"\nSaved summary -> {summary_path}")
try:
    from IPython.display import display
    display(summary[["subject","roi_key","lh_vertices","rh_vertices","bilat_vertices","n_images","saved"]]
            .sort_values(["roi_key","subject"]))
except Exception:
    pass